# **Cancellation Prediction**

## Objectives

* Extend the preprocessing pipeline with transformations/encoding appropriate for tree-based classification
* Train and evaluate a classification model to predict cancellation
* Answer Business Requirement 2: *TCS Hotels wants a machine learning model capable of predicting the likelihood of a booking cancellation, accessed through an operational dashboard that supports the reservations team in three ways: a risk report of upcoming arrivals, individual reservation search and a prospective booking risk assessor*

## Inputs

* Split datasets X_train, X_test, y_train and y_test from "outputs/ml_pipeline/preprocessing/" 
* Preprocessing pipeline "outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl"

## Outputs

* Classification preprocessing pipeline saved to "outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl"
* Classification modelling pipeline saved to "outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl"
* Trained classification model for Business Requirement 2, to be integrated into the operational dashboard

## Additional Comments

* ⚠️ TBC ⚠️


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect'

---

## Load Data

In [3]:
import pandas as pd

X_train = pd.read_csv("outputs/ml_pipeline/preprocessing/X_train.csv")
X_test = pd.read_csv("outputs/ml_pipeline/preprocessing/X_test.csv")
y_train = pd.read_csv("outputs/ml_pipeline/preprocessing/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/ml_pipeline/preprocessing/y_test.csv").squeeze()

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(95344, 29) (23836, 29) (95344,) (23836,)


---

# Add Model Specific Preprocessing

**Classification Pipeline Actions**

| **Feature** | **Prediction model actions** | **Experimental prediction model alternatives** |
| --- | --- | --- |
| is_canceled | Target only | SMOTE |
| lead_time | | Log/skew transformation; binning |
| arrival_date_month | One-hot | Cyclical encoding |
| arrival_date_week_number | Exclude | Cyclical encoding |
|stays_in_weekend_nights||binning|
|stays_in_week_nights||binning|
|adults||binning|
|children||binary, binning|
|babies||binary, binning|
| country | Ordinal encoding | Frequency encoding; target encoding |
| previous_cancellations | | Binary |
| previous_bookings_not_canceled | | Log/skew transformation; binning |
| booking_changes | | Binary |
| agent | | Frequency encoding; target encoding; exclude |
| days_in_waiting_list | | Binary; log/skew transformation |
| adr | | Log/skew transformation; binning |
| required_car_parking_spaces | | Binary |
| total_of_special_requests | | Binning; binary |


In [4]:
drop = ["arrival_date_week_number"]
ordinal = ["country"]
one_hot = ["arrival_date_month"]

* Load and test the preprocessing pipeline

In [5]:
data = X_train.copy()
data.head(3)

,hotel,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,11,2015,December,49,5,2,1,2,0.0,...,A,0,No Deposit,350.0,NaN,0,Transient-Party,68.0,0,0
1,Resort Hotel,2,2017,May,19,7,1,0,1,0.0,...,D,0,No Deposit,NaN,195.0,0,Group,45.0,0,0
2,City Hotel,9,2016,September,38,11,1,0,2,0.0,...,G,0,No Deposit,NaN,NaN,0,Transient,0.0,0,2


In [ ]:
import joblib
from utils.custom_transformers import undefined_meal

preprocessing_pipeline = joblib.load("outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")
pipeline_step1 = preprocessing_pipeline.fit_transform(data)
pipeline_step1.shape


(95344, 55)

In [7]:
pipeline_step1.head()

,lead_time,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,...,assigned_room_type_B,assigned_room_type_F,assigned_room_type_H,assigned_room_type_K,assigned_room_type_I,deposit_type_No Deposit,deposit_type_Non Refund,customer_type_Transient-Party,customer_type_Group,customer_type_Transient
0,11,December,49,5,2,1,2,0.0,0,ESP,...,0,0,0,0,0,1,0,1,0,0
1,2,May,19,7,1,0,1,0.0,0,PRT,...,0,0,0,0,0,1,0,0,1,0
2,9,September,38,11,1,0,2,0.0,0,PRT,...,0,0,0,0,0,1,0,0,0,1
3,19,February,9,27,1,3,3,0.0,0,PRT,...,0,0,0,0,0,1,0,0,0,1
4,218,June,23,8,0,3,2,0.0,0,GBR,...,0,0,0,0,0,1,0,0,0,1


In [8]:
pipeline_step1["country"].isnull().sum()

np.int64(0)

* Drop `arrival_date_week_number` 

In [9]:
from feature_engine.selection import DropFeatures

drop_transformer = DropFeatures(features_to_drop=drop)
pipeline_step2 = drop_transformer.fit_transform(pipeline_step1)
pipeline_step2.shape

(95344, 54)

* Encode `country`

In [10]:
from feature_engine.encoding import OrdinalEncoder

encoder = OrdinalEncoder(encoding_method="arbitrary", variables=ordinal)
pipeline_step3 = encoder.fit_transform(pipeline_step2)
pipeline_step3["country"].head(3)

0    0
1    1
2    1
Name: country, dtype: int64

In [11]:
pipeline_step3["country"].isnull().sum()

np.int64(0)

* One-hot encode `arrival_date_month`

In [12]:
from feature_engine.encoding import OneHotEncoder

encoder = OneHotEncoder(variables=one_hot)
pipeline_step4 = encoder.fit_transform(pipeline_step3)
pipeline_step4.shape

(95344, 65)

* Build the model specific preprocessing pipeline and test

In [13]:
from feature_engine.encoding import RareLabelEncoder
from sklearn.pipeline import Pipeline

def classification_preprocessing_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", preprocessing_pipeline),
        ("DropFeatures", DropFeatures(features_to_drop=drop)),
        ("RareLabelEncoder", RareLabelEncoder(tol=0.01, variables=ordinal)),
        ("OrdinalEncoder", OrdinalEncoder(encoding_method="arbitrary", variables=ordinal, ignore_format=True)),
        ("OneHotEncoder", OneHotEncoder(variables=one_hot))        
    ])

    return pipeline_base

* Unplanned RareLabelEncoder added because the OrdinalEncoder added NaN values on the unseen test set

In [15]:
test_df = X_train.copy()
classification_model_preprocessing_pipeline = classification_preprocessing_pipeline()
pipeline_test = classification_model_preprocessing_pipeline.fit_transform(test_df)
pipeline_test.shape

(95344, 65)

In [16]:
classification_model_preprocessing_pipeline.named_steps

{'Preprocessing': Pipeline(steps=[('DropFeatures',
                  DropFeatures(features_to_drop=['company',
                                                 'arrival_date_year'])),
                 ('FunctionTransformer',
                  FunctionTransformer(func=<function undefined_meal at 0x717fd127cf40>)),
                 ('ArbitraryNumberImputer',
                  ArbitraryNumberImputer(arbitrary_number=0, variables='agent')),
                 ('CategoricalImputer',
                  CategoricalImputer(imputation_method='frequent',
                                     variables='country')),
                 ('Winsorizer',
                  Winsorizer(capping_method='iqr', fold=1.5,
                             variables=['lead_time', 'adr',
                                        'stays_in_weekend_nights',
                                        'stays_in_week_nights'])),
                 ('OneHotEncoder',
                  OneHotEncoder(drop_last=True,
                      

In [17]:
pipeline_test["country"].isnull().sum()

np.int64(0)

* Test that there are no remaining NaN values in `country`

In [18]:
encoding_error_test = X_test.copy()
error_test = classification_model_preprocessing_pipeline.transform(encoding_error_test)
error_test["country"].isnull().sum()

np.int64(0)

---

## Classification modelling pipeline

In [19]:
from sklearn.preprocessing import StandardScaler

def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("Scaler", StandardScaler()),
        ("model", model)
    ])

    return pipeline_base

* Model selection

In [20]:
# Code adapted from the Churnometer walkthrough

from sklearn.model_selection import GridSearchCV
import numpy as np


class ModelComparison:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = classification_pipeline(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

* Use standard hyperparameters to find the most suitable model

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

models_search = {
    "LogisticRegression": LogisticRegression(random_state=0, max_iter=500),  # Default value of 100 was insufficient 
    "DecisionTreeClassifier": DecisionTreeClassifier(random_state=0),
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
    "XGBClassifier": XGBClassifier(random_state=0),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=0),
}

params_search = {
    "LogisticRegression": {},
    "DecisionTreeClassifier": {},
    "RandomForestClassifier": {},
    "XGBClassifier": {},
    "GradientBoostingClassifier": {},
}

In [22]:
from sklearn.metrics import make_scorer, recall_score
search = ModelComparison(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)


Running GridSearchCV for LogisticRegression 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for DecisionTreeClassifier 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for RandomForestClassifier 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for XGBClassifier 

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for GradientBoostingClassifier 

Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [23]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

,estimator,min_score,mean_score,max_score,std_score
2,RandomForestClassifier,0.797114,0.802388,0.807894,0.003535
3,XGBClassifier,0.793577,0.802218,0.812535,0.006863
1,DecisionTreeClassifier,0.789332,0.798483,0.80249,0.00513
4,GradientBoostingClassifier,0.720045,0.726812,0.732918,0.004473
0,LogisticRegression,0.615167,0.621414,0.632852,0.006067


* Multiple algorithms were evaluated using identical processing and 5-fold ross-validation
* Recall was used as the primary optimisation metric because identifying cancellations is the main business requirement
* LogisticRegression was tested to evaluate linear relationships, and to provide a baseline. The task is binary classification so there was little expectation of a strong performance and this was backed up by the results.
* Tree-based methods performed substantially better with both RandomForest and XGBoost achieving 0.802 which meets the target recall of 0.8 set out in the business understanding. These will be carried forward for tuning and evaluation. There is nothing to separate the two models at this stage in terms of mean_score, their min/max scores are also comparable at RF: 0.797/0.807 and XGB: 0.794/0.813
* DecisionTree was very close at 0.798 and with a max_score of 0.802 will be carried forward to provide a baseline.
* std_score for all models tested was less than 0.01 suggesting stability across the folds in all models.
* GradientBoosting and LogisticRegression are dropped at this stage since neither achieved the target recall value with their max_scores (0.733 and 0.633 repectively)
 

---

## Hyperparameter tuning 

* remove scaling from the pipeline since LogisticRegression discounted

In [25]:
def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", model)
    ])

    return pipeline_base

**RandomForest**

In [34]:
models_search = {
    "RandomForestClassifier": RandomForestClassifier(random_state=0),
}

params_search = {
    "RandomForestClassifier": {
        "model__max_depth": [None, 10, 20, 30]
    },
}

In [35]:
search = ModelComparison(models=models_search, params=params_search)
search.fit(X_train, y_train,
           scoring =  make_scorer(recall_score, pos_label=1),
           n_jobs=-1, cv=5)


Running GridSearchCV for RandomForestClassifier 

Fitting 5 folds for each of 4 candidates, totalling 20 fits


In [37]:
rf_results = pd.DataFrame(
    grid_search_pipelines["RandomForestClassifier"].cv_results_
)

rf_results = (
    rf_results[
        [
            "param_model__max_depth",
            "mean_test_score",
            "std_test_score",
            "rank_test_score",
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
)

rf_results

,param_model__max_depth,mean_test_score,std_test_score,rank_test_score
0,None,0.802218,0.003356,1
3,30,0.801511,0.003575,2
2,20,0.765265,0.003368,3
1,10,0.607973,0.005648,4


In [41]:
def parameter_search(model, parameter_settings):

    best_params = {}

    history = []

    for parameter, values in parameter_settings.items():

        search_params = best_params.copy()
        search_params[parameter] = values

        gs = GridSearchCV(
            classification_pipeline(model),
            param_grid=search_params,
            cv=5,
            scoring="recall",
            n_jobs=-1,
        )

        gs.fit(X_train, y_train.values.ravel())

        best_params[parameter] = [gs.best_params_[parameter]]

        history.append(
            {
                "Parameter": parameter,
                "Best value": gs.best_params_[parameter],
                "Best recall": gs.best_score_,
            }
        )

    return best_params, pd.DataFrame(history)

In [ ]:
model = RandomForestClassifier(random_state=0) 

params_options = { 
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1,2,4,8],
    "model__n_estimators": [100, 300, 500],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", "log2"]
    }

best_params, history = parameter_search(model, params_options)
print(best_params)
history

---

# Push files to Repo

* In case you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create here your folder
  # os.makedirs(name='')
except Exception as e:
  print(e)
